### Model Comparison

Candidate models x feature sets with cross-validation

In [2]:
# import libraries
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [3]:
# load the cleaned/transformed data and split into train/test sets
df = pd.read_csv("../data/stars_transformed.csv")
y = df["Type"]
X = df.drop(columns=["Type", "Type_Name"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)     # test set stays untouched until notebook 05

(192, 17) (48, 17)


In [4]:
# define feature sets for model comparison (the experiment axis to compare the effect of adding more features)
color_cols = [c for c in X.columns if c.startswith("color_")]
base = ["log_Temperature", "log_L"]

FEATURE_SETS = {
    "A: log T + log L":            base,
    "B: + log R":                  base + ["log_R"],
    "C: + A_M (numeric only)":     base + ["log_R", "A_M"],
    "D: C + spectral":             base + ["log_R", "A_M", "Spectral_Ordinal"],
    "E: D + color":                base + ["log_R", "A_M", "Spectral_Ordinal"] + color_cols,
}

In [5]:
# define candidate models for comparison
MODELS = {
    "Baseline (majority)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "KNN (k=5)":           make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "SVM (RBF)":           make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(300, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

In [6]:
# define cross-validation strategy and scoring metrics
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scoring = {"acc": "accuracy", "f1": "f1_macro", "logloss": "neg_log_loss"}

rows = []
for fs_name, cols in FEATURE_SETS.items():
    for m_name, model in MODELS.items():
        r = cross_validate(model, X_train[cols], y_train, cv=cv, scoring=scoring)
        rows.append({"feature set": fs_name, "model": m_name,
                     "accuracy": r["test_acc"].mean(), "acc_std": r["test_acc"].std(),
                     "macro_f1": r["test_f1"].mean(),
                     "log_loss": -r["test_logloss"].mean()})
results = pd.DataFrame(rows)
results.pivot(index="model", columns="feature set", values="accuracy").round(3)

c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\skle

feature set,A: log T + log L,B: + log R,C: + A_M (numeric only),D: C + spectral,E: D + color
model,,,,,
Baseline (majority),0.156,0.156,0.156,0.156,0.156
Decision Tree,0.830,0.943,0.998,0.989,0.997
Gradient Boosting,0.789,0.952,0.989,0.985,0.984
KNN (k=5),0.787,0.950,0.987,0.976,0.921
Logistic Regression,0.792,0.940,0.970,0.962,0.961
Random Forest,0.806,0.959,1.000,1.000,0.998
SVM (RBF),0.791,0.954,0.982,0.971,0.907


In [ ]:
# compute differences to identify change in accuracy in percentage points when moving from one feature set to the next
acc = results.pivot(index="model", columns="feature set", values="accuracy")
(acc.diff(axis=1).iloc[:, 1:] * 100).round(1)

feature set,B: + log R,C: + A_M (numeric only),D: C + spectral,E: D + color
model,,,,
Baseline (majority),0.0,0.0,0.0,0.0
Decision Tree,11.3,5.5,-0.9,0.8
Gradient Boosting,16.3,3.7,-0.3,-0.1
KNN (k=5),16.3,3.8,-1.2,-5.5
Logistic Regression,14.7,3.0,-0.7,-0.1
Random Forest,15.3,4.1,0.0,-0.2
SVM (RBF),16.3,2.8,-1.1,-6.4


In [7]:
# sort results by accuracy (descending) and log loss (ascending)
results.sort_values(["accuracy", "log_loss"], ascending=[False, True]).head(15).round(3)

,feature set,model,accuracy,acc_std,macro_f1,log_loss
19,C: + A_M (numeric only),Random Forest,1.000,0.000,1.000,0.022
26,D: C + spectral,Random Forest,1.000,0.000,1.000,0.035
33,E: D + color,Random Forest,0.998,0.007,0.998,0.063
18,C: + A_M (numeric only),Decision Tree,0.998,0.007,0.998,0.076
32,E: D + color,Decision Tree,0.997,0.009,0.997,0.114
20,C: + A_M (numeric only),Gradient Boosting,0.989,0.015,0.988,0.024
25,D: C + spectral,Decision Tree,0.989,0.013,0.988,0.414
16,C: + A_M (numeric only),KNN (k=5),0.987,0.020,0.987,0.072
27,D: C + spectral,Gradient Boosting,0.985,0.015,0.985,0.026
34,E: D + color,Gradient Boosting,0.984,0.017,0.984,0.026


### Tuning and Selection

In [8]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

cols = FEATURE_SETS["C: + A_M (numeric only)"]          # use the set you chose
grids = {
    "Logistic Regression": {"logisticregression__C": [0.1, 1, 10, 100]},
    "SVM (RBF)": {"svc__C": [1, 10, 100], "svc__gamma": ["scale", 0.1, 0.01]},
    "Random Forest": {"n_estimators": [200, 500], "max_depth": [None, 5, 10],
                      "min_samples_leaf": [1, 2, 4]},
}
inner = StratifiedKFold(5, shuffle=True, random_state=1)
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7)

for name, grid in grids.items():
    gs = GridSearchCV(MODELS[name], grid, cv=inner, scoring="f1_macro")
    s = cross_val_score(gs, X_train[cols], y_train, cv=outer, scoring="f1_macro")
    print(f"{name}: nested macro-F1 = {s.mean():.3f} +/- {s.std():.3f}")

Logistic Regression: nested macro-F1 = 0.990 +/- 0.019


c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\skle

SVM (RBF): nested macro-F1 = 0.991 +/- 0.013
Random Forest: nested macro-F1 = 0.998 +/- 0.007


### Tuning on log loss

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold, cross_validate

cols = ["log_Temperature", "log_L", "log_R", "A_M"]          # feature set C

grids = {
    "Random Forest": (RandomForestClassifier(random_state=42),
        {"n_estimators": [200, 400], "max_depth": [None, 6], "min_samples_leaf": [1, 3]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
        {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}),
    "SVM (RBF)": (make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
        {"svc__C": [1, 10, 100], "svc__gamma": ["scale", 0.1, 0.3]}),
}
inner = StratifiedKFold(5, shuffle=True, random_state=1)
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7)

tuned = {}
for name, (est, grid) in grids.items():
    gs = GridSearchCV(est, grid, cv=inner, scoring="neg_log_loss", n_jobs=-1)
    r = cross_validate(gs, X_train[cols], y_train, cv=outer, n_jobs=1,
                       scoring={"acc": "accuracy", "logloss": "neg_log_loss", "f1": "f1_macro"})
    gs.fit(X_train[cols], y_train)
    tuned[name] = gs.best_estimator_
    print(f"{name}: acc={r['test_acc'].mean():.3f}±{r['test_acc'].std():.3f} "
          f"logloss={-r['test_logloss'].mean():.3f} best={gs.best_params_}")

Random Forest: acc=1.000±0.000 logloss=0.022 best={'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Gradient Boosting: acc=0.997±0.009 logloss=0.005 best={'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}


c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\skle

SVM (RBF): acc=0.993±0.012 logloss=0.150 best={'svc__C': 10, 'svc__gamma': 'scale'}


c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### Noise Robustness

In [11]:
from sklearn.base import clone

def stress_cv(model, X, y, dex=0.05, mag=0.3, seed=0):
    rng = np.random.default_rng(seed)
    accs = []
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42).split(X, y):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        Xn = X.iloc[te].copy()
        for c in ["log_Temperature", "log_L", "log_R"]:
            Xn[c] += rng.normal(0, dex, len(Xn))       # noise in log10 units
        Xn["A_M"] += rng.normal(0, mag, len(Xn))
        accs.append((m.predict(Xn) == y.iloc[te]).mean())
    return np.mean(accs), np.std(accs)

for name, model in tuned.items():
    print(name, "mild:", stress_cv(model, X_train[cols], y_train),
                "heavy:", stress_cv(model, X_train[cols], y_train, 0.15, 1.0))

Random Forest mild: (np.float64(0.989608636977058), np.float64(0.018493617774098064)) heavy: (np.float64(0.9529014844804319), np.float64(0.0386431774115305))
Gradient Boosting mild: (np.float64(0.9825461088618982), np.float64(0.020721390739463223)) heavy: (np.float64(0.940710751237067), np.float64(0.0340077996497731))


c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\skle

SVM (RBF) mild: (np.float64(0.9930724246513719), np.float64(0.01148940509028567)) heavy: (np.float64(0.9669365721997301), np.float64(0.027946412166147364))


c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Samsung\AppData\Local\Programs\Python\Python315\Lib\site-packages\skle

### Saving the model

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier

clean = pd.read_csv("../data/stars_clean.csv")
RAW = ["Temperature", "L", "R", "A_M"]

# Same split settings as the comparison, so the same 48 stars stay held out
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    clean[RAW], clean["Type"], test_size=0.2, stratify=clean["Type"], random_state=42)

final = make_pipeline(
    ColumnTransformer([("log", FunctionTransformer(np.log10), ["Temperature", "L", "R"])],
                      remainder="passthrough"),
    RandomForestClassifier(n_estimators=300, random_state=42),
)
final.fit(Xr_train, yr_train)

Path("../models").mkdir(exist_ok=True)
joblib.dump(final, "../models/star_classifier.joblib")
print("Saved. Training stars:", len(Xr_train), "| held-out:", len(Xr_test))